# Wikipedia edits: Structured Streaming explorations

The topic `wikistreams` contains the events sent by the two producers:
- `wikistream_producer.py` (course demo): edits on `fr.wikipedia.org`, replayed from the start of the stream history
- `wikistream_producer_filtered.py`: live edits on `en.wikipedia.org`, articles only (namespace 0)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               LongType, IntegerType, BooleanType)

In [2]:
spark = SparkSession.builder \
    .appName("wikistream_explorations") \
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

## Streaming dataframe

In [3]:
raw_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()

schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("namespace", IntegerType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

edits_df = raw_df \
    .select(F.from_json(F.col("value").cast("string"), schema).alias("data")) \
    .select("data.*") \
    .withColumn("event_time", F.timestamp_seconds("timestamp")) \
    .withColumn("size_diff", F.col("length.new") - F.col("length.old"))

Each query below reads everything available in Kafka (`trigger(availableNow=True)`), writes the result to an in-memory table, then stops so the table can be displayed.

In [4]:
def run_query(df, name):
    query = df.writeStream \
        .format("memory") \
        .queryName(name) \
        .outputMode("complete") \
        .trigger(availableNow=True) \
        .start()
    query.awaitTermination()
    return spark.table(name)

## 1. Events per wiki

In [5]:
per_wiki = edits_df.groupBy("wiki").agg(
    F.count("*").alias("edits"),
    F.min("event_time").alias("first_event"),
    F.max("event_time").alias("last_event"),
    F.round(F.avg(F.col("bot").cast("int")) * 100, 1).alias("bot_pct")
)

run_query(per_wiki, "per_wiki").show(truncate=False)

+------+-----+-------------------+-------------------+-------+
|wiki  |edits|first_event        |last_event         |bot_pct|
+------+-----+-------------------+-------------------+-------+
|enwiki|478  |2026-10-07 09:45:12|2026-10-07 09:55:06|3.6    |
|frwiki|8447 |2026-09-30 09:16:09|2026-09-30 15:28:30|17.7   |
+------+-----+-------------------+-------------------+-------+



8,447 French edits replayed from the stream history (30 September, 09:16 to 15:28, so about 6 hours of edits sent in 10 minutes) and 478 live English article edits (10 minutes). Bots make 17.7 % of the French edits and 3.6 % of the English ones (the English producer keeps only articles).

## 2. Tumbling windows: edits every 5 minutes, bots vs humans

In [6]:
tumbling = edits_df \
    .groupBy(F.window("event_time", "5 minutes"), "wiki", "bot") \
    .agg(F.count("*").alias("edits"),
         F.round(F.avg("size_diff"), 1).alias("avg_size_diff")) \
    .select(F.col("window.start").alias("window_start"),
            F.col("window.end").alias("window_end"),
            "wiki", "bot", "edits", "avg_size_diff")

run_query(tumbling, "tumbling") \
    .orderBy("wiki", "window_start", "bot") \
    .show(40, truncate=False)

+-------------------+-------------------+------+-----+-----+-------------+
|window_start       |window_end         |wiki  |bot  |edits|avg_size_diff|
+-------------------+-------------------+------+-----+-----+-------------+
|2026-10-07 09:45:00|2026-10-07 09:50:00|enwiki|false|208  |-363.2       |
|2026-10-07 09:45:00|2026-10-07 09:50:00|enwiki|true |8    |13.0         |
|2026-10-07 09:50:00|2026-10-07 09:55:00|enwiki|false|252  |117.6        |
|2026-10-07 09:50:00|2026-10-07 09:55:00|enwiki|true |9    |28.7         |
|2026-10-07 09:55:00|2026-10-07 10:00:00|enwiki|false|1    |1.0          |
|2026-09-30 09:15:00|2026-09-30 09:20:00|frwiki|false|86   |91.3         |
|2026-09-30 09:15:00|2026-09-30 09:20:00|frwiki|true |16   |383.8        |
|2026-09-30 09:20:00|2026-09-30 09:25:00|frwiki|false|94   |74.7         |
|2026-09-30 09:20:00|2026-09-30 09:25:00|frwiki|true |24   |216.1        |
|2026-09-30 09:25:00|2026-09-30 09:30:00|frwiki|false|89   |148.2        |
|2026-09-30 09:25:00|2026

Tumbling windows do not overlap: each edit is counted in exactly one 5 minute window. There are about 100 French edits and 220 to 260 English article edits per window. The average size change moves a lot between windows because one big edit is enough to change it (English window 09:45: -363 bytes on average, because of one removal of 106,471 bytes, see part 4).

## 3. Overlapping windows: 10 minutes sliding every 5 minutes

In [7]:
sliding = edits_df \
    .groupBy(F.window("event_time", "10 minutes", "5 minutes"), "wiki") \
    .agg(F.count("*").alias("edits"),
         F.sum(F.when(F.col("size_diff") > 0, F.col("size_diff"))
               .otherwise(0)).alias("bytes_added"),
         F.sum(F.when(F.col("size_diff") < 0, -F.col("size_diff"))
               .otherwise(0)).alias("bytes_removed")) \
    .select(F.col("window.start").alias("window_start"),
            F.col("window.end").alias("window_end"),
            "wiki", "edits", "bytes_added", "bytes_removed")

run_query(sliding, "sliding") \
    .orderBy("wiki", "window_start") \
    .show(40, truncate=False)

+-------------------+-------------------+------+-----+-----------+-------------+
|window_start       |window_end         |wiki  |edits|bytes_added|bytes_removed|
+-------------------+-------------------+------+-----+-----------+-------------+
|2026-10-07 09:40:00|2026-10-07 09:50:00|enwiki|216  |35624      |111062       |
|2026-10-07 09:45:00|2026-10-07 09:55:00|enwiki|477  |72226      |117769       |
|2026-10-07 09:50:00|2026-10-07 10:00:00|enwiki|262  |36603      |6707         |
|2026-10-07 09:55:00|2026-10-07 10:05:00|enwiki|1    |1          |0            |
|2026-09-30 09:10:00|2026-09-30 09:20:00|frwiki|102  |16172      |2183         |
|2026-09-30 09:15:00|2026-09-30 09:25:00|frwiki|220  |32549      |6353         |
|2026-09-30 09:20:00|2026-09-30 09:30:00|frwiki|219  |47254      |16271        |
|2026-09-30 09:25:00|2026-09-30 09:35:00|frwiki|211  |56837      |16655        |
|2026-09-30 09:30:00|2026-09-30 09:40:00|frwiki|217  |53183      |16757        |
|2026-09-30 09:35:00|2026-09

A new 10 minute window starts every 5 minutes, so each edit is counted in 2 windows. Compared to the tumbling windows, the activity curve is smoother. The French windows starting at 10:45 and 10:50 have about 100,000 bytes added, twice the usual level of about 50,000.

## 4. Size of the edits

In [8]:
size_types = edits_df \
    .withColumn("change", F.when(F.col("size_diff") > 0, "addition")
                .when(F.col("size_diff") < 0, "removal")
                .otherwise("same size")) \
    .groupBy("wiki", "change") \
    .agg(F.count("*").alias("edits"),
         F.round(F.avg(F.abs("size_diff")), 1).alias("avg_bytes"),
         F.max(F.abs("size_diff")).alias("max_bytes"))

run_query(size_types, "size_types").orderBy("wiki", "change").show()

+------+---------+-----+---------+---------+
|  wiki|   change|edits|avg_bytes|max_bytes|
+------+---------+-----+---------+---------+
|enwiki| addition|  356|    202.9|     6035|
|enwiki|  removal|   97|   1214.1|   106471|
|enwiki|same size|   25|      0.0|        0|
|frwiki| addition| 6011|    383.8|   121236|
|frwiki|  removal| 2042|    320.1|    47762|
|frwiki|same size|  394|      0.0|        0|
+------+---------+-----+---------+---------+



In [9]:
biggest_pages = edits_df \
    .groupBy("wiki", "title") \
    .agg(F.count("*").alias("edits"),
         F.sum("size_diff").alias("total_size_diff"))

run_query(biggest_pages, "biggest_pages") \
    .orderBy(F.desc(F.abs("total_size_diff"))) \
    .show(10, truncate=False)

+------+--------------------------------------------+-----+---------------+
|wiki  |title                                       |edits|total_size_diff|
+------+--------------------------------------------+-----+---------------+
|frwiki|Liste des sénateurs français de 2026 à 2029 |6    |133383         |
|enwiki|Textile industry in Bangladesh              |1    |-106471        |
|frwiki|Rallye de Grande-Bretagne 1970              |5    |57791          |
|frwiki|Mégalopole du delta de la Rivière des Perles|2    |49869          |
|frwiki|Discussion utilisateur:Livfornow            |1    |-47762         |
|frwiki|Projet:Wiki Loves Women/Les femmes sportives|1    |25341          |
|frwiki|Discussion:Thélyson Orélien                 |43   |22319          |
|frwiki|Utilisateur:Rais67/Histoire de Djerba       |6    |21668          |
|frwiki|Utilisateur:Semeldicitur/Brouillon          |3    |-18396         |
|frwiki|Vallée de Chevreuse                         |15   |17650          |
+------+----

Most edits add content: 71 % of the French edits (6,011 / 8,447) and 74 % of the English ones (356 / 478). Removals are less frequent but can be very large: one single edit removed 106,471 bytes from the English article "Textile industry in Bangladesh".

## 5. Most active users per wiki

In [10]:
users = edits_df.groupBy("wiki", "user", "bot").count()

run_query(users, "users") \
    .orderBy(F.desc("count")) \
    .show(10, truncate=False)

+------+-----------+-----+-----+
|wiki  |user       |bot  |count|
+------+-----------+-----+-----+
|frwiki|OrlodrimBot|true |544  |
|frwiki|CodexBot   |true |452  |
|frwiki|CodexBot2  |true |337  |
|frwiki|Polmars    |false|231  |
|frwiki|Baidax     |false|103  |
|frwiki|R3L3ct3ur49|false|91   |
|frwiki|FDo64      |false|83   |
|frwiki|Pld        |false|79   |
|frwiki|Kailingkaz |false|78   |
|frwiki|ListeriaBot|true |72   |
+------+-----------+-----+-----+
only showing top 10 rows


The three most active French accounts are bots: OrlodrimBot (544), CodexBot (452) and CodexBot2 (337), together 16 % of the French edits. The most active human account made 231 edits.